# ML baseline on track 1: how much do surface cues recover?

The classical machine-learning baseline for Assignment 1, built as an ablation of what the EDA found: direction lives in surface asymmetry.

**Systems**

| System | Features | Model |
|---|---|---|
| `cues+logreg` | 17 numeric cues from `dico_nli.features`: lengths, length difference and ratio, word overlap, containment in each direction, shared first and last word, span containment, digits | logistic regression, balanced class weights |
| `tfidf+logreg` | word 1- and 2-grams of `premise [SEP] hypothesis` | logistic regression, balanced class weights |
| `both+logreg` | the cues and the TF-IDF features side by side | logistic regression, balanced class weights |
| `cues+gbdt` | the 17 cues | gradient-boosted trees |

**Protocol, fixed before any run**

1. Hyperparameters are scikit-learn defaults with `C = 1.0`; nothing is tuned.
2. Stability comes from 5-fold cross-validation on train, grouped by source pair so both directions of a pair stay in the same fold; every fold is scored with the official scorer.
3. Each system is then fitted on the full train set and scored once on dev.
4. The confusion matrix is drawn for the system with the best mean cross-validated weighted F1, a choice made on train only. The coefficient figure always shows `cues+logreg`, because the question it answers is which cues carry direction.

**Where the results go**

| Output | Path |
|---|---|
| Cross-validation folds and summary | `experiments/2026-09-29-ml-baseline-track1/tables/cv_folds.csv`, `cv_summary.csv` |
| Dev scores, pair consistency, confusion matrix, cue weights | `experiments/2026-09-29-ml-baseline-track1/tables/` |
| Dev predictions and scorer reports | `experiments/2026-09-29-ml-baseline-track1/predictions/`, `scores/<system>/` |
| Figures | `experiments/2026-09-29-ml-baseline-track1/figures/*.pdf`, with `.png` previews |
| Run record and results table | `experiments/2026-09-29-ml-baseline-track1/config.json`, `results.md` |
| Written findings | `experiments/2026-09-29-ml-baseline-track1/findings.md` |
| Fold-level files | `artifacts/ml-baseline-track1/` (ignored by git) |

**How to run**

```bash
PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/03_ml_baseline_track1.ipynb
```

Needs the task repository cloned at `datasets/dico-nli/`. Takes about 90 seconds on a laptop CPU.

## 1. Setup

Imports, the four configurations and the run parameters.

In [ ]:
from datetime import date

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from dico_nli.baselines.ml import MLBaseline, MLConfig, cross_validate
from dico_nli.data import DATA_ROOT, LABELS, REPO_ROOT, load_reference
from dico_nli.experiments import experiment_dir, write_config, write_results_table
from dico_nli.plotting import INK, INK_SECONDARY, SEQUENTIAL, figure, paper_theme, save_figure
from dico_nli.scoring import score, write_predictions

TRACK = 1
SEED = 13
N_SPLITS = 5
RUN_DATE = date(2026, 9, 29)
OVERWRITE = True
CONFIGS = [
    MLConfig(features="cues", model="logreg", seed=SEED),
    MLConfig(features="tfidf", model="logreg", seed=SEED),
    MLConfig(features="both", model="logreg", seed=SEED),
    MLConfig(features="cues", model="gbdt", seed=SEED),
]
SHORT = {"EQUIVALENCE": "Equiv.", "FORWARD_ENTAILMENT": "Forward",
         "BACKWARD_ENTAILMENT": "Backward", "NEGATIVE_OTHER": "Neg./other"}
ORDER = list(LABELS)
paper_theme()

## 2. Data

Train and dev, the dev reference file for the scorer, and the run folder. The cross-validation folds are written under `artifacts/`, which git ignores.

In [ ]:
train = load_reference(TRACK, "train")
dev = load_reference(TRACK, "dev")
gold = DATA_ROOT / "dev" / f"dico_nli_dev_track{TRACK}_reference.csv"
out = experiment_dir(f"ml-baseline-track{TRACK}", day=RUN_DATE, overwrite=OVERWRITE)
tables, figures = out / "tables", out / "figures"
tables.mkdir(exist_ok=True)
work = REPO_ROOT / "artifacts" / f"ml-baseline-track{TRACK}"
work.mkdir(parents=True, exist_ok=True)


def show(fig, stem):
    pdf_path, png_path = save_figure(fig, figures / stem)
    display(Image(filename=str(png_path)))


print(f"train {len(train)}  dev {len(dev)}  ->  {out.relative_to(REPO_ROOT)}")

## 3. Cross-validation on train, grouped by source pair

Five folds, seed 13. A fold never splits a reversible pair, so SoftCons and HardCons are computable inside each fold. The per-fold scores are saved; the summary below is the mean and standard deviation per system.

In [ ]:
fold_frames = []
for config in CONFIGS:
    folds = cross_validate(config, train, n_splits=N_SPLITS, seed=SEED, work_dir=work / "cv" / config.name)
    fold_frames.append(folds.assign(system=config.name))
cv = pd.concat(fold_frames, ignore_index=True)
cv.to_csv(tables / "cv_folds.csv", index=False)
cv_summary = (cv.groupby("system", sort=False)[["weighted_f1", "soft_cons", "hard_cons"]]
                .agg(["mean", "std"]).round(4))
cv_summary.columns = [f"{metric}_{stat}" for metric, stat in cv_summary.columns]
cv_summary.to_csv(tables / "cv_summary.csv")
best_by_cv = cv_summary["weighted_f1_mean"].idxmax()
print("best by cross-validated weighted F1:", best_by_cv)
cv_summary

## 4. Fit on the full train set, score dev once

One fit per system on all 3,042 training instances, one prediction file per system, one call to the official scorer each.

In [ ]:
models, rows, dev_predictions = {}, {}, {}
for config in CONFIGS:
    model = MLBaseline(config).fit(train)
    predictions = model.predict(dev)
    prediction_path = write_predictions(out / "predictions" / f"{config.name}.csv", predictions)
    rows[config.name] = score(gold, prediction_path, out / "scores" / config.name)
    models[config.name], dev_predictions[config.name] = model, predictions

dev_table = pd.DataFrame({name: {"weighted_f1": s.weighted_f1, "soft_cons": s.soft_cons, "hard_cons": s.hard_cons}
                          for name, s in rows.items()}).T
dev_table.to_csv(tables / "dev_scores.csv")
dev_table.round(4)

## 5. Pair-level view of consistency

Over the reversible dev pairs, the scorer counts how many were predicted compatibly in both directions (the SoftCons numerator) and how many were fully correct (the HardCons numerator). The difference is the number of pairs where a system was self-consistent and wrong in both directions.

In [ ]:
import json

pairs = []
for name, s in rows.items():
    report = json.loads(s.report_path.read_text(encoding="utf-8"))["consistency"]
    pairs.append({"system": name, "reversible_pairs": report["reversible_pairs"],
                  "soft_consistent_pairs": report["soft_consistent_pairs"],
                  "hard_correct_pairs": report["hard_correct_pairs"]})
pairs = pd.DataFrame(pairs).set_index("system")
pairs["consistent_but_wrong"] = pairs["soft_consistent_pairs"] - pairs["hard_correct_pairs"]
pairs.to_csv(tables / "dev_pair_consistency.csv")
pairs

## 6. Figures

**Figure 1.** The three official scores per system on dev.

In [ ]:
metrics = [("weighted_f1", "Weighted F1", SEQUENTIAL[1]), ("soft_cons", "SoftCons", SEQUENTIAL[3]), ("hard_cons", "HardCons", SEQUENTIAL[5])]
systems = list(dev_table.index)
fig, ax = figure(columns=1, height=2.4)
y = np.arange(len(systems))
for k, (column, name, colour) in enumerate(metrics):
    values = dev_table[column].to_numpy()
    ax.barh(y + (k - 1) * 0.26, values, height=0.24, color=colour, label=name)
    for yy, v in zip(y + (k - 1) * 0.26, values):
        ax.text(v + 0.01, yy, f"{v:.2f}", va="center", fontsize=6.5, color=INK_SECONDARY)
ax.set_yticks(y)
ax.set_yticklabels(systems)
ax.invert_yaxis()
ax.set_xlim(0, 1.12)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.grid(False, axis="y")
ax.grid(True, axis="x")
ax.set_title("ML baselines on track 1 dev")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, handlelength=1.0)
show(fig, "f1_dev_scores")

**Figure 2.** Confusion matrix on dev for the system chosen by cross-validation. Rows are gold labels, columns are predictions.

In [ ]:
gold_labels = {i.instance_id: i.label for i in dev}
chosen = dev_predictions[best_by_cv]
confusion = pd.crosstab(pd.Series([gold_labels[k] for k in chosen], name="gold"),
                        pd.Series(list(chosen.values()), name="predicted")).reindex(index=ORDER, columns=ORDER).fillna(0).astype(int)
confusion.to_csv(tables / f"dev_confusion_{best_by_cv}.csv")

fig, ax = figure(columns=1, height=2.6)
sns.heatmap(confusion, ax=ax, cmap=sns.light_palette(SEQUENTIAL[5], as_cmap=True), annot=True, fmt="d",
            cbar=False, linewidths=1.5, linecolor="white", annot_kws={"size": 7},
            xticklabels=[SHORT[l] for l in ORDER], yticklabels=[SHORT[l] for l in ORDER])
ax.set_xlabel("predicted")
ax.set_ylabel("gold")
ax.set_title(f"{best_by_cv} on dev")
ax.grid(False)
ax.tick_params(length=0)
show(fig, "f2_confusion")

**Figure 3.** Standardised logistic-regression weights of `cues+logreg`, one column per label. A positive weight pushes towards the label; the direction cues (length difference, containment in each direction) should show opposite signs for FORWARD and BACKWARD.

In [ ]:
coefficients = models["cues+logreg"].cue_coefficients()
coefficients.to_csv(tables / "cue_coefficients.csv", index=False)
matrix = coefficients.pivot(index="cue", columns="label", values="coefficient").reindex(columns=ORDER)
order = matrix.abs().max(axis=1).sort_values(ascending=False).index
matrix = matrix.loc[order]

fig, ax = figure(columns=1, height=3.6)
limit = float(np.abs(matrix.to_numpy()).max())
sns.heatmap(matrix, ax=ax, cmap=sns.diverging_palette(250, 15, s=80, l=50, as_cmap=True), center=0,
            vmin=-limit, vmax=limit, annot=True, fmt=".1f", annot_kws={"size": 6}, cbar=False,
            linewidths=1.0, linecolor="white", xticklabels=[SHORT[l] for l in ORDER])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Cue weights of cues+logreg (standardised)")
ax.grid(False)
ax.tick_params(length=0)
show(fig, "f3_cue_weights")

## 7. Run record

`config.json` records the seed, the data commit, this repository's commit and the command, plus the settings of every system and which one cross-validation chose. `results.md` holds the dev table in the form the report cites.

In [ ]:
write_config(
    out, seed=SEED, track=TRACK,
    command="PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/03_ml_baseline_track1.ipynb",
    extra={"n_splits": N_SPLITS, "systems": [c.name for c in CONFIGS], "C": 1.0,
           "class_weight": "balanced", "tfidf": "word 1-2 grams, sublinear tf",
           "gbdt": "HistGradientBoosting, 300 iterations, learning rate 0.05",
           "selection": "best mean cross-validated weighted F1 on train", "best_by_cv": best_by_cv,
           "train_size": len(train), "dev_size": len(dev)},
)
print(write_results_table(out, rows, title=f"ML baselines, track {TRACK} dev").read_text(encoding="utf-8"))